# LIBERO — **옛 50ep 정리 + 로그에서 500ep eval_info 복구**

옛 **50ep** eval 잔재(`seedN/actions`·`seedN/videos`·`seedN/eval_info.json`)를 통째로 지워
깔끔한 **500ep**(`seedN/rep0/…`)만 남기고, eval 은 끝났는데 `eval_info.json` 이 안 남은
것(seed1·3)을 **로그에서 SR 복구**한다.

판별(실제 로그 확인): LIBERO-10 = 10 task → **정상 500ep = overall n_ep 5000**, 옛 50ep = 500.
action_logs 파일 수도 새 ~500 / 옛 ~50 (10 task 가 한 폴더에 덮어써 마지막 task 것만 남음).
→ **eval 출력 디렉토리(E) 단위**로 새/옛을 나눈다: 옛 E 의 `actions`·`videos` 폴더와 `eval_info.json` 제거.

- ⚠️ seed1 의 현재 eval_info 는 **옛 50ep 것** → 지우고 새로 복구.
- 순서: **인벤토리 → ① 옛것 제거 → ② 로그 복구**. 둘 다 dry-run→EXECUTE.


In [ ]:
import sys, json, re, shutil
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)

TASK   = 'libero_10'
MODELS = ['act', 'acm', 'acm2', 'mosaic', 'bimamba', 'bimamba_s7']
SEEDS  = [0, 1, 2, 3]
N_TASKS  = 10                     # LIBERO-10 = 10 task
PER_TASK = 500                    # 정상 = task당 500ep → overall n_ep = 5000. 옛 50ep = 500.
MIN_NEW_EP  = N_TASKS * PER_TASK // 2   # eval_info overall n_ep 이 이 값(2500) 이상 = 새
MIN_NEW_ACT = PER_TASK // 2             # action_logs episode 파일 수 이 값(250) 이상 = 새

EVAL_ROOT = cf.OUTPUT_BASE / 'eval_clean' / TASK
LOG_DIR   = cf.OUTPUT_BASE / '_logs'
print('eval:', EVAL_ROOT, '| logs:', LOG_DIR)
print(f'판별: 새(500ep)=overall n_ep>={MIN_NEW_EP} 또는 action>={MIN_NEW_ACT} / 옛(50ep)=그 미만')

## 인벤토리 — eval 디렉토리별 새/옛 + 제거/복구 대상


In [ ]:
# ── eval 출력 디렉토리(E) 단위로 새/옛 판별. E = actions/ 또는 eval_info.json 의 부모 ──
#    옛 50ep 는 seed 바로 밑(seedN/actions·videos·eval_info), 새 500ep 는 seedN/rep0/ 아래.
def n_ep_of(inf):
    try:
        return int(json.loads(inf.read_text()).get('overall', {}).get('n_episodes') or 0)
    except Exception:
        return 0

def act_count(E):
    al = E / 'actions' / 'action_logs'
    return len(list(al.glob('episode_*.pt'))) if al.is_dir() else 0

plan_remove, need_recover, need_fresh = [], [], []
print(f"{'model':<11}{'seed':>4}   NEW(500ep) / OLD(50ep)  [eval디렉토리: act개수, info n_ep]")
print('-' * 80)
for tag in MODELS:
    for s in SEEDS:
        seed_dir = EVAL_ROOT / tag / f'seed{s}'
        if not seed_dir.is_dir():
            print(f'{tag:<11}{s:>4}   (폴더 없음)'); need_fresh.append((tag, s)); continue
        eval_dirs = set()
        for a in seed_dir.rglob('actions'):
            if a.is_dir():
                eval_dirs.add(a.parent)
        for inf in seed_dir.rglob('eval_info.json'):
            eval_dirs.add(inf.parent)
        if not eval_dirs:
            print(f'{tag:<11}{s:>4}   (eval 결과 없음)'); need_fresh.append((tag, s)); continue
        row_new, row_old = [], []
        has_new, has_new_info, new_noinfo = False, False, []
        for E in sorted(eval_dirs):
            cnt = act_count(E)
            inf = E / 'eval_info.json'
            ne = n_ep_of(inf) if inf.exists() else None
            is_new = (ne is not None and ne >= MIN_NEW_EP) or cnt >= MIN_NEW_ACT
            rel = str(E.relative_to(seed_dir))
            loc = '(seed 직속)' if rel == '.' else rel
            desc = f'{loc}[act {cnt}, info {ne if ne is not None else "-"}]'
            if is_new:
                has_new = True; row_new.append(desc)
                if inf.exists() and (ne or 0) >= MIN_NEW_EP:
                    has_new_info = True
                else:
                    new_noinfo.append(E)
            else:
                row_old.append(desc)
                for name in ('actions', 'videos'):    # 옛 폴더 통째로
                    if (E / name).is_dir():
                        plan_remove.append(E / name)
                if inf.exists():                        # 옛 eval_info
                    plan_remove.append(inf)
        print(f'{tag:<11}{s:>4}   NEW={row_new or "-"}  OLD={row_old or "-"}')
        if has_new and not has_new_info and new_noinfo:
            need_recover.append((tag, s, new_noinfo[0]))
        if not has_new:
            need_fresh.append((tag, s))

# 옛 50ep 로그(파일명에 '50ep')도 제거 대상
old_logs = [p for p in LOG_DIR.glob('*.log') if '50ep' in p.name] if LOG_DIR.is_dir() else []

print('\n' + '=' * 80)
print('■ 제거할 옛 50ep 경로(actions/videos/eval_info):', len(plan_remove), '개')
print('■ 제거할 옛 50ep 로그(*50ep*.log):', len(old_logs), '개')
print('■ 로그에서 eval_info 복구 대상:', [f'{t}/s{s}' for t, s, _ in need_recover] or '없음')
print('■ 새 500ep eval 필요(새 데이터 없음):', [f'{t}/s{s}' for t, s in need_fresh] or '없음')

## ① 옛 50ep 제거 — `actions`·`videos` 폴더 + `eval_info.json` + 옛 `*50ep*.log` (dry-run → EXECUTE=True)
**새(500ep, rep0)·`job__*rep0.log` 는 절대 안 건드림.**


In [ ]:
# ── ① 옛 50ep 제거: actions·videos 폴더 + eval_info.json + 옛 50ep 로그 ── EXECUTE=True ──
EXECUTE = False

if not plan_remove and not old_logs:
    print('제거할 옛 50ep 없음.')
else:
    print(f'{"제거 실행" if EXECUTE else "DRY-RUN(안 지움)"} — 결과 {len(plan_remove)}개 + 로그 {len(old_logs)}개:')
    for p in plan_remove:
        kind = 'DIR ' if p.is_dir() else 'FILE'
        print(f'   {kind} {p.relative_to(EVAL_ROOT)}')
        if EXECUTE:
            shutil.rmtree(p) if p.is_dir() else p.unlink()
    for lp in old_logs:
        print(f'   LOG  _logs/{lp.name}')
        if EXECUTE:
            lp.unlink()
    print('\n' + ('제거 완료.' if EXECUTE else '확인됐으면 EXECUTE=True 로 다시 실행. (새 500ep·job 로그는 안 건드림)'))

## ② 로그에서 500ep eval_info 복구 (dry-run → EXECUTE=True)
결과 못 찾으면 **깊은 진단**(마커·마지막 runSR·끝줄)으로 원인 표시: `Aggregated Metrics`/`End of eval` 마커가 없으면 그 eval 은 **미완/크래시** → 재eval 필요.


In [ ]:
# ── ② 로그에서 새(500ep) SR 복구 → eval_info.json 생성 ── 확인 후 EXECUTE=True ──
EXECUTE = False

def logs_for(tag, seed):
    # 로그 파일명 규칙: ..{tag}__seed{N}.. (예: job__bimamba__seed0__rep0.log)
    if not LOG_DIR.is_dir():
        return []
    key = f'{tag}__seed{seed}'
    return sorted(p for p in LOG_DIR.glob('*.log') if key in p.name)

_NE = re.compile(r"n_episodes['\"]?\s*[:=]\s*([0-9]+)")
_PC = re.compile(r"pc_success['\"]?\s*[:=]\s*([0-9]+\.?[0-9]*)")

def scan_log(lp):
    # 로그의 모든 (pc_success, n_episodes) 쌍. 순서/인접 무관: n_episodes 주변 ±300자에서 pc_success 를 찾음.
    txt = lp.read_text(errors='ignore')
    pairs = []
    for m in _NE.finditer(txt):
        ne = int(m.group(1))
        seg = txt[max(0, m.start() - 300): m.end() + 100]
        pcs = _PC.findall(seg)
        if pcs:
            pairs.append((float(pcs[-1]), ne))     # n_episodes 에 가장 가까운 pc_success
    return pairs

def inspect_log(lp):
    # 결과가 없을 때 왜 없는지: 크기 · 마커 유무 · 마지막 running SR · 끝줄
    txt = lp.read_text(errors='ignore')
    marks = [m for m in ['End of eval', 'Aggregated Metrics', 'running_success_rate',
                         'Traceback', 'Killed', 'out of memory', 'Error']
             if m in txt]
    rs = re.findall(r"running_success_rate['\"]?\s*[:=]\s*['\"]?([0-9.]+)", txt)
    lines = [l for l in txt.splitlines() if l.strip()]
    tail = lines[-1][:150] if lines else '(빈 파일)'
    return f'{len(txt)//1024//1024}MB · 마커={marks or "없음"} · 마지막 runSR={rs[-1] if rs else "-"}% · 끝줄: {tail}'

def new_result_from_logs(tag, seed):
    best, diag = None, []
    for lp in logs_for(tag, seed):
        pairs = scan_log(lp)
        nes = sorted({ne for _, ne in pairs})
        diag.append((lp, f'n_ep 후보: {nes[-6:] if nes else "없음"}'))
        for sr, ne in pairs:
            if ne >= MIN_NEW_EP:
                best = (sr, ne, lp)               # 새(500ep) 중 마지막(최근)
    return best, diag

if not need_recover:
    print('복구 대상 없음.')
else:
    print(f'{"복원" if EXECUTE else "DRY-RUN"} — 로그→eval_info:')
    for tag, s, E in need_recover:
        res, diag = new_result_from_logs(tag, s)
        if res is None:
            print(f'   {tag}/seed{s}: 새(n_ep>={MIN_NEW_EP}) 결과 못 찾음. 매칭 로그:')
            for lp, d in diag:
                print(f'        - {lp.name}  ({d})')
                if '없음' in d and '50ep' not in lp.name:      # 결과 없는 새(job) 로그 = 진단
                    print(f'            → {inspect_log(lp)}')
            if not diag:
                print(f'        - (매칭 로그 없음 — {LOG_DIR}/*{tag}__seed{s}*.log 확인)')
            continue
        sr, ne, lp = res
        target = E / 'eval_info.json'
        print(f'   {tag}/seed{s}: SR={sr:.2f}% n_ep={ne} (per-task {ne//N_TASKS}) → {target.relative_to(EVAL_ROOT)}  [{lp.name}]')
        if EXECUTE and not target.exists():
            target.write_text(json.dumps(
                {'overall': {'pc_success': sr, 'n_episodes': ne}, '_recovered_from_log': lp.name}, indent=2))
    print('\n' + ('복원 완료 → eval_final 재실행하면 반영.' if EXECUTE else '확인됐으면 EXECUTE=True.'))

## ③ 재eval 대상 정리 (미완/누락) — ⚠️ 500ep ≈ 40시간/개


In [ ]:
# ── ③ 미완/누락 eval 재실행 대상 정리 ── ⚠️ LIBERO 500ep = 5000 에피소드 ≈ 40시간/개 ──
#    로그에 완료 결과가 없는 것(need_recover)은 미완 → 재eval. 학습 안 된 건 먼저 retrain.
EXECUTE = False

def is_trained(tag, s):
    return (cf.v23.last_ckpt_step(cf.OUTPUT_BASE / 'train' / TASK / tag / f'seed{s}') or 0) >= cf.CKPT_STEP

cand = {(t, s) for t, s, _ in need_recover} | {(t, s) for t, s in need_fresh}
reeval = sorted((t, s) for t, s in cand if is_trained(t, s))          # 학습됨 → 바로 재eval
need_train = sorted((t, s) for t, s in cand if not is_trained(t, s))  # 미학습 → retrain 먼저

print('■ 재eval (학습됨 + 유효 500ep 없음):', reeval or '없음')
print('■ 먼저 retrain 필요 (체크포인트 없음/부족):', need_train or '없음')
print(f'\n⚠️ 500ep eval 은 개당 ~40시간(5000 에피소드). {len(reeval)}개 → 여러 GPU 로 나누고,')
print('   중간에 안 죽는 환경(nohup/충분한 walltime)에서. 진행중이던 거면 그냥 두고 기다려도 됨.')
print('   (실제 실행은 여기서 EXECUTE=True, 또는 libero/eval_node1_4gpu·eval_node2_4gpu 사용)')
if EXECUTE and reeval:
    gpus = cf.available_gpus()
    print(f'\n재eval 시작: {reeval}  (GPU {gpus})')
    cf.run_libero_eval_jobs(reeval, gpus=gpus, n_episodes=cf.EVAL_N_EP)   # 없는 것만 실행(끝난 건 skip)